In [ ]:


using Gen
using Plots
using Statistics
using Distributions 

# Define the model with dynamic eta sampling
@gen function eight_school_model(sigma)
    mu ~ normal(0, 1)     # Sample mu from a normal distribution
    tau  ~ normal(2, 5)   # Sample tau from a Half-Cauchy distribution
    
    # Dynamic eta values sampled from normal distributions
    list_of_Eta = [{(:eta, i)} ~ normal(0, 1) for i=1:length(sigma)]
    
    for i in 1:length(sigma)  # Loop over 5 iterations
        # Calculate theta based on mu, tau, and eta
        theta = mu + tau * list_of_Eta[i]
        
        # Sample obs from a normal distribution with mean theta and standard deviation sigma[i]
         {(:y, i)} ~ normal(theta, sigma[i])
    end

    
    
    
end

function multi_variable_metropolis(trace, model, sigma, observations, eps)
    # Extract current values of parameters
    mu_current = get_choices(trace)[:mu]
    tau_current = get_choices(trace)[:tau]
    eta_current = [get_choices(trace)[(:eta, i)] for i in 1:length(sigma)]
    
    
    # Propose new values for mu, tau, and eta
    mu_proposed = mu_current + eps * randn()
    tau_proposed = tau_current + eps * randn()
    eta_proposed = [eta_current[i] + eps * randn() for i in 1:length(sigma)]
    
     # Create a temporary choice map with the proposed values
    temp_cm = choicemap(
        (:mu => mu_proposed),
        (:tau => tau_proposed)
    )
   
    # Add proposed eta values to the choice map
    for i in 1:length(sigma)
        
        temp_cm[(:eta, i)] = eta_proposed[i]
        # print("hoi") 
    end

    # Use Gen.update to get the updated trace after applying the proposed values
    (proposed_trace, _, _) = Gen.update(
        trace,                # Current trace                # Generative model
        (sigma,), (),            # Arguments for the generative function
        temp_cm               # Temporary choice map with proposed values
    )

    ###
    current_score = get_score(trace)
    #print(current_score)
    
    proposed_score = get_score(proposed_trace)
    # print(proposed_score)
    # Compute the acceptance ratio using the gradients
    acceptance_ratio = min(1.0, exp(proposed_score - current_score))
    # print(acceptance_ratio)
    # Accept or reject based on the acceptance ratio
    if rand() < acceptance_ratio
        return (proposed_trace, 1)
    else
        return (trace, 0)  # Keep the current state if not accepted
    end
end

function do_inference(model, sigma, y_obs, num_iters, eps)
    observations = choicemap()
    for (i, y) in enumerate(y_obs)
        observations[(:y, i)] = y
    end

    (trace, _) = generate(model, (sigma,), observations)
    accepted = 0
    mu_samples = []
    tau_samples = []

    # Store sampled values at each iteration
    for _ in 1:num_iters
        (trace, accepted_this_iter) = multi_variable_metropolis(trace, model, sigma, observations, eps)
        #accepted += accepted_this_iter
        
        # Store samples
        final_choices = get_choices(trace)
        push!(mu_samples, final_choices[:mu])
        push!(tau_samples, final_choices[:tau])
    end

    #acceptance_rate = accepted / num_iters  # Compute acceptance rate

    return (mu_samples, tau_samples)
end





sigma = [15, 10, 16, 11, 9, 11, 10, 18]
y_obs = [28, 8, -3, 7, -1, 1, 18, 12]
num_iters = 1000  # Number of iterations for Metropolis-Hastings
eps = 0.1         # Step size for proposals

(mu_samples, tau_samples) = do_inference(eight_school_model, sigma, y_obs, num_iters, eps)
print(mean(mu_samples))

